# WIMB: train your own model on a free Colab GPU

**How to run it**
1. Check the top right says **T4**. If it doesn't: **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Runtime → Run all**.
3. In the first minute Google asks to connect **Google Drive**. Click **Connect** and **Allow**
   (the trained model is saved there).
4. Keep this tab open. It takes about 30–40 minutes.
5. When it's done: **File → Save a copy in GitHub** → repository `pratik9yadav/yok`,
   branch `main`, file path `train_colab.ipynb` → **OK**.
   Then tell Claude it's done. Claude reads the results from GitHub.

**What it does:** takes a free model (Qwen3 4B), tests it on shop calls, trains it on
400 examples, tests it again, then tests the saved Ollama file too. Two test sets: 30 calls
and 56 harder ones written in styles the training data does not have. The model goes to Google Drive.

It also runs on Kaggle's free GPU: `run_on_kaggle.py` starts it there with no clicks.

In [ ]:
#@title 1. Check the GPU
import os, shutil, subprocess, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle gives two GPUs; training uses one
# nvidia-smi is not always installed, so also ask torch (in its own process: nothing is imported here yet).
cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                      capture_output=True, text=True).stdout.strip() == "True"
if not (cuda or shutil.which("nvidia-smi")):
    # What the machine has, to see why there is no GPU (no secret values printed).
    print("nvidia devices:", [d for d in os.listdir("/dev") if d.startswith("nvidia")])
    print("Kaggle settings:", {k: v for k, v in os.environ.items()
                               if k.startswith("KAGGLE") and any(w in k for w in ("ACCELERATOR", "GPU", "MACHINE", "RUN_TYPE"))})
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU > Save, then Run all. "
                       "Kaggle: verify your phone, then turn on GPU T4 once in the notebook editor (Settings > Accelerator).")
print(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
                     capture_output=True, text=True).stdout or "GPU found")

In [ ]:
#@title 2. Install the training tools (about 3–5 minutes)
!pip install -q unsloth

In [ ]:
#@title 3. Settings (and connect Google Drive now, so nothing waits for you later)
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
EPOCHS = 2  #@param {type:"integer"}
# v2: training data built from real products, prices and spellings, plus 151 real Hindi answers.
# v1: the first 400 made-up examples (kept to compare against).
TRAIN_SET = "v2"  #@param ["v2", "v1"]
OUTPUT_NAME = "wimb-shop-model"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
MAKE_OLLAMA_FILE = True  #@param {type:"boolean"}
# This notebook is set up for Qwen3 instruct models (their chat format is used below).

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/tmp" if ON_KAGGLE else "/content"  # big temporary files (Kaggle saves all of /kaggle/working)
if ON_KAGGLE:
    SAVE_DIR = f"/kaggle/working/{OUTPUT_NAME}"  # shown in the notebook's Output tab
elif SAVE_TO_DRIVE:
    SAVE_DIR = f"/content/drive/MyDrive/{OUTPUT_NAME}"
    from google.colab import drive
    drive.mount("/content/drive")
else:
    SAVE_DIR = None

In [ ]:
#@title 4. Training data and scoring (hidden; built from data/ and score.py)
"""Shared by the notebook and the tests: the task's prompt, and how an answer is scored.

The task: read what a shopkeeper said on a phone call and return JSON with four fields.
"""

import json
import re
import time

SYSTEM = (
    "You read what a shopkeeper said on a phone call about a product. "
    "Reply with only a JSON object with these keys: "
    "in_stock (true, false or null), price_inr (a number or null), "
    "delivery (true, false or null), alternative (text or null). "
    "price_inr is the price of the product that was asked for. "
    "Use null when the shopkeeper did not say."
)

FIELDS = ("in_stock", "price_inr", "delivery", "alternative")


def user_message(product: str, said: str) -> str:
    return f"Product: {product}\nShopkeeper: {said}"


def answer_text(answer: dict) -> str:
    """The exact text the model is trained to reply with."""
    return json.dumps({k: answer[k] for k in FIELDS}, ensure_ascii=False)


def parse(text: str) -> dict | None:
    """The last JSON object in the reply that has exactly the four keys, or None.

    Every "{" is tried, so braces inside a value ("M-seal {2 pack}") or other text around the
    answer do not hide it.
    """
    decoder = json.JSONDecoder()
    found = None
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text, match.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and set(value) == set(FIELDS):
            found = value
    return found


def _tokens(text: str) -> set[str]:
    """Lower-case words, with "40mm" read as "40 mm" and punctuation dropped."""
    text = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", text.lower())
    return set(re.sub(r"[-_.,;:!?()\[\]{}\"'/]", " ", text).split())


def same(field: str, got, want) -> bool:
    if want is None or got is None:
        return got is None and want is None
    if field in ("in_stock", "delivery"):
        return isinstance(got, bool) and got == want  # 1 and 0 are not true and false
    if field == "price_inr":
        if isinstance(got, bool):
            return False
        try:
            return float(got) == float(want)
        except (TypeError, ValueError):
            return False
    # alternative: the wanted words must all be there, with at most two extra ("40 mm padlock").
    if not isinstance(got, str):
        return False
    have, need = _tokens(got), _tokens(want)
    return bool(have) and need <= have and len(have - need) <= 2


def score(replies: list[str], examples: list[dict]) -> dict:
    """Per-field accuracy, all-four-right accuracy, and the mistakes.

    A reply with no valid answer object (or a missing or extra key) gets every field wrong.
    """
    right = {f: 0 for f in FIELDS}
    all_right = json_ok = 0
    mistakes = []
    for reply, ex in zip(replies, examples, strict=True):
        got = parse(reply)
        if got is not None:
            json_ok += 1
        wrong = list(FIELDS) if got is None else [f for f in FIELDS if not same(f, got[f], ex["answer"][f])]
        for f in FIELDS:
            right[f] += f not in wrong
        if wrong:
            mistakes.append({"said": ex["said"], "want": ex["answer"], "reply": reply, "wrong": wrong})
        else:
            all_right += 1
    n = len(examples)
    return {
        "examples": n,
        "json_ok": json_ok / n,
        "all_correct": all_right / n,
        **{f: right[f] / n for f in FIELDS},
        "mistakes": mistakes,
    }


KEYS = ("all_correct", "json_ok", *FIELDS)


def evaluate(ask, sets: dict[str, list[dict]], deadline: float | None = None) -> dict:
    """Run ask(product, said) over every test set and score it. Every reply is kept.

    After `deadline` (a time.time() value) no more questions are asked; what was answered is
    scored, and "of" says how many there were.
    """
    out = {}
    for name, rows in sets.items():
        replies = []
        for ex in rows:
            if deadline is not None and time.time() > deadline:
                break
            replies.append(ask(ex["product"], ex["said"]))
        done = rows[: len(replies)]
        scored = score(replies, done) if done else {"examples": 0, "mistakes": []}
        out[name] = {**scored, "replies": replies, "of": len(rows)}
    return out


def show(label: str, results: dict, max_mistakes: int = 8) -> None:
    for name, res in results.items():
        n = res["examples"]
        if not n:
            print(f"== {label} [{name}]: no calls answered")
            continue
        partial = f" (only {n} of {res['of']} answered)" if n < res["of"] else ""
        print(f"== {label} [{name}]: all 4 fields right in {res['all_correct']:.0%} of {n} calls{partial}")
        print("   " + "   ".join(f"{k} {res[k]:.0%}" for k in KEYS[1:]))
        for m in res["mistakes"][:max_mistakes]:
            print("\n   said: ", m["said"])
            print("   want: ", answer_text(m["want"]))
            print("   reply:", m["reply"][:200])
            print("   wrong:", ", ".join(m["wrong"]))


def results_payload(model: str, epochs: int, before: dict, after: dict) -> dict:
    """What results.json holds: scores, mistakes and every raw reply, so scores can be recomputed."""
    sets = {}
    for name, a in after.items():
        b = before[name]
        sets[name] = {
            "examples": a["of"],
            "before": {k: round(b[k], 3) for k in KEYS},
            "after": {k: round(a[k], 3) for k in KEYS},
            "mistakes_before": b["mistakes"],
            "mistakes_after": a["mistakes"],
            "replies_before": b["replies"],
            "replies_after": a["replies"],
        }
    return {"model": model, "epochs": epochs, "sets": sets}


def make_modelfile(gguf_name: str) -> str:
    """An Ollama Modelfile whose template is exactly the text the model was trained on.

    Unsloth's own Qwen3 template differs a little (a blank line after the system message, and
    thinking-mode syntax that needs a newer Ollama), so it is not used.
    """
    return (
        f"FROM ./{gguf_name}\n"
        'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
        '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
        '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        "PARAMETER temperature 0\n"
        f'SYSTEM """{SYSTEM}"""\n'
    )


TRAIN_V2 = [json.loads(line) for line in r'''
{"product": "Splendor brake shoe", "said": "haan bolo, hai bhai, home delivery hai, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, haan, stock mein hai, 220 rupaye ka hai, home delivery hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "Yes, available. It's 80 rupees", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes sir, we have. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते। उसकी जगह 3.15 mfd capacitor दे सकते हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3.15 mfd capacitor"}}
{"product": "silicone sealant", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, मिल जाएगा। 160 लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "aahe na, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan bola, aahe na, 10 piece aahet, 80 rupaye honar, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Not sure, I'll have to check the godown. Come before 10", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 16 piece aahet, vees rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, ho, milel, pannas rupaye", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. No, we don't have it. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "khatam ho gaya, uski jagah 7W LED bulb de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7W LED bulb"}}
{"product": "4 inch door hinge", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, didshe rupaye", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 110 lagega", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है। 160 रुपये का है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Not today, it comes tomorrow. We don't deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aahe na, 11 piece aahet, rate 390 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, available. That will be Rs 160", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ, है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, stock mein hai, 1600 rupaye ka hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "available hai, rate 310 hai", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan bhai, hai bhai, 15 piece pade hain, rate 360 hai", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां बोला. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. हो, आहे. 950 ला पडेल. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan ji, mil jayega, rate 380 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए। वो हम नहीं रखते। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan, stock mein hai, 5 piece pade hain, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aahe na, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ji boliye, nahi hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, tell me. Yes, we have it", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji boliye, available hai, 18 hi bache hain, ₹15, dukaan 8 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, aahe na, 14 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan bhai, abhi nahi hai, parso aayega, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, we have it. Price is 35. We can deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "stock madhe aahe, 50 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, haan, stock mein hai, 160 lagega, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan bhai, ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "थांबा, बघून सांगतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Hello, who is this?. Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "available hai, dhai sau rupaye final, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "available hai, 2 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉक में है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. पन्नास ला पडेल. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji haan, rakha hai, 20 lagega, delivery ho jayegi, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hello, kaun?, hai bhai, 410 rupaye final", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, we have it. Price is 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई। 1100 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, ghari pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मालक नाहीत, नंतर फोन करा. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। ख़त्म हो गया। घर पर भेज देंगे। दुकान 8 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ji haan, rakha hai, 130 rupaye final, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bola, aahe na, rate 1600 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "hello, kaun?, thodi der baad call karo, dukaan 8 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bhai, abhi stock mein nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aamhi te thevat nahi, tyachya aivaji CPVC elbow deu shakto, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "CPVC elbow"}}
{"product": "12V 7Ah battery", "said": "आहे ना. 9 पीस आहेत. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan, stock mein hai, 2150 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi nahi hai, parso aayega, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आम्ही ते ठेवत नाही. local putty आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local putty"}}
{"product": "Activa air filter", "said": "है भाई। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan ji, mil jayega", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "available hai, 15 piece pade hain, 35 ka padega", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give CPVC elbow instead. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "silicone sealant", "said": "ji boliye, haan ji, mil jayega, 180 rupaye final, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन?। रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो, कोण?. नाही आहे. Pulsar 180 spark plug आहे, चालेल का?. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Pulsar 180 spark plug"}}
{"product": "Activa air filter", "said": "nahi hai, ladka bhej deta hun, dukaan 10 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, 7 piece aahet, 380 rupaye honar", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ho, milel, 1750 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai, haan hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, uski jagah local clutch plate de sakte hain, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "Yes, available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 5 hi bache hain, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji boliye, available hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "haan hai, 6 hi bache hain, ₹140", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला. आहे ना. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan hai, 830 lagega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "ji boliye, pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, thread seal tape hai, chalega?, delivery nahi karte, aake le jao, dukaan 9 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "thread seal tape"}}
{"product": "50 mm padlock", "said": "ji boliye, haan, stock mein hai, 17 hi bache hain, 150 ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 14 piece aahet, 50 rupaye, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai, parso aayega, 10 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo, hai bhai, 400 rupaye final", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, आहे. 1450 रुपये", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bhai, available hai, 610 lagega", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 5 पीस पड़े हैं। 1600 रुपये का है। घर पर भेज देंगे। दुकान 7 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "It's in stock. We have 14 pieces. Price is 180. I'll send it with our boy. Come before 10", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi stock mein nahi hai, uski jagah CPVC elbow de sakte hain, ladka bhej deta hun, 9 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "Activa air filter", "said": "नहीं है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "आहे ना. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "सध्या नाही, उद्या येईल. paint roller आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "2 inch paint brush", "said": "abhi nahi hai, parso aayega, paint roller hai, chalega?, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "fan capacitor 2.5 mfd", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "hello, kon?, sampla, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "abhi stock mein nahi hai, 7W LED bulb hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "7W LED bulb"}}
{"product": "12V 7Ah battery", "said": "It's in stock. It's 1050 rupees. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "aahe na, shambhar rupaye", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "We don't keep that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म हो गया। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji boliye, abhi nahi hai, parso aayega, uski jagah 60 mm padlock de sakte hain, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "bike headlight bulb", "said": "Hello?. It's in stock. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, 130 rupaye", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "stock madhe aahe, 410 la padel", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, ji haan, rakha hai, 330 ka padega", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan aahe, 19 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. Price is 290. Pickup only", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bolo, haan ji, mil jayega, 120 lagega", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. It's in stock. That will be Rs 100. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "It's in stock. 70 rupees. I'll send it with our boy. Come before 9", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 rupaye honar", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan hai, 30 lagega, ghar pe bhej denge, 7 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है। paint roller है, चलेगा?। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "paint roller"}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, mil jayega, rate 380 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, we have it. That will be Rs 1800. We don't deliver", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, आहे. 8 पीस आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 210 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, rate 1300 hai", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bola, thamba, baghun sangto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "nahi milega bhai, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, we have it. That will be Rs 240", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, tyachya aivaji wall putty 40 kg deu shakto, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "wall putty 40 kg"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए। स्टॉक में है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji haan, rakha hai, 75 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bhai, nahi milega bhai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, hai bhai, ₹1500", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. हो, आहे. 13 पीस आहेत. 120 रुपये. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan, stock mein hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give 3/4 inch elbow instead. Pickup only", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3/4 inch elbow"}}
{"product": "9W LED bulb", "said": "आहे ना. 3 पीस आहेत. रेट 140 आहे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 30 rupaye honar, delivery deto", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bola, ho aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan ji, mil jayega, 80 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji boliye, hai bhai, rate pachaas hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नहीं है। उसकी जगह 12V 9Ah battery दे सकते हैं। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12V 9Ah battery"}}
{"product": "teflon tape", "said": "ji boliye, haan ji, mil jayega, rate 30 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, thamba, baghun sangto, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "woh hum nahi rakhte, LED headlight bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 3 hi bache hain, ₹80, home delivery hai", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan bolo, haan ji, mil jayega, rate 50 hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "hai bhai, 14 piece pade hain, delivery ho jayegi, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "It's in stock. That will be Rs 1700", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नाही आहे. त्याच्याऐवजी 3.15 mfd capacitor देऊ शकतो. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "3.15 mfd capacitor"}}
{"product": "Fevicol SH 1 kg", "said": "bola, ho aahe, 420 rupaye honar", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "स्टॉक में है। रेट 2200 है। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hai bhai, 7 hi bache hain, ₹120, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "ho, milel, 9 ch urle aahet, 70 rupaye honar", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 1150 लगेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। ख़त्म हो गया। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "है भाई। 230 लगेगा। दुकान 10 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho, milel, 270 rupaye honar", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock madhe aahe, adichshe rupaye honar, ghari pathavto", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, rate 90 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "nahi milega bhai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 5 piece pade hain, 220 lagega", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, है। 13 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bolo, hai bhai, 1450 rupaye ka hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we have it. Price is 45. Pickup only", "answer": {"in_stock": true, "price_inr": 45, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bolo, nahi hai, uski jagah 60 mm padlock de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "60 mm padlock"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "khatam ho gaya, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो, कौन?। हाँ, है। 20 पीस पड़े हैं। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes sir, we have. We have 4 pieces. That will be Rs 90. Pickup only", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. 80 रुपये. घरी पाठवतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan ji, mil jayega, 730 ka padega", "answer": {"in_stock": true, "price_inr": 730, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "सध्या नाही, उद्या येईल. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji haan, rakha hai, rate 240 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, है। 120 लगेगा। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "available hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, halogen bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "halogen bulb"}}
{"product": "1 inch PVC elbow", "said": "हां बोला. स्टॉकमध्ये आहे. तीस रुपये", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, available. 65 rupees", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "It's in stock. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji haan, rakha hai, 100 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. It's in stock. We have 9 pieces. It's 220 rupees. We can deliver", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "nahi milega bhai, local brake shoe hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "4 socket extension board", "said": "hai bhai, 520 ka padega", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan aahe, 10 rupaye, yeun nyava lagel, 9 chya aat ya", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "pata nahi, godown mein dekhna padega, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Hello?. No, we don't have it. We have local seal kit, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Activa clutch plate", "said": "malik abhi nahi hain, baad mein phone karo, 9 baje ke pehle aana", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "malak nahit, nantar phone kara, 9 chya aat ya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, 1500 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Yes, tell me. We don't keep that. I can give local brake shoe instead. Home delivery is available. Come before 8", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "bike headlight bulb", "said": "hai bhai, 7 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Yes, tell me. Yes sir, we have. Price is 350", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Sorry, finished. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "khatam ho gaya, 2.25 mfd capacitor rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.25 mfd capacitor"}}
{"product": "Pulsar 150 spark plug", "said": "आम्ही ते ठेवत नाही. local spark plug आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "9W LED bulb", "said": "Hello?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai, ₹110, delivery nahi hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, haan aahe, 170 rupaye, ghari pathavto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 120 rupaye, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "available hai, 17 piece pade hain, rate 400 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Sorry, finished. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan bola, udya yeil, aaj nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "sampla, 8 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello, kaun?, abhi stock mein nahi hai, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 6 piece pade hain, 70 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, kaun?, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "abhi nahi hai, parso aayega, uski jagah 2.25 mfd capacitor de sakte hain, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2.25 mfd capacitor"}}
{"product": "50 mm padlock", "said": "हाँ बोलिए। स्टॉक में है। 2 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi stock mein nahi hai, local seal kit rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Pulsar 150 spark plug", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. स्टॉकमध्ये आहे. 10 पीस आहेत. तीस रुपये. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hello?. Yes, we have it. Only 6 left. It's 1100 rupees", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, है। 17 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, local air filter hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local air filter"}}
{"product": "Pulsar 150 spark plug", "said": "udya yeil, aaj nahi, Pulsar 180 spark plug aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180 spark plug"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ji haan, rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hello, kaun?, woh hum nahi rakhte, delivery nahi hai, dukaan 8 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello, kon?, sampla, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ji haan, rakha hai, 760 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bhai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। रेट 90 है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ख़त्म हो गया। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला. नाही आहे. त्याच्याऐवजी 2.5 sq mm wire देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, haan ji, mil jayega, 70 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hello, kaun?, nahi hai, 12W LED bulb hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "woh hum nahi rakhte, 1 sq mm wire rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "8 mm drill bit", "said": "haan bhai, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "sampla, dukan 7 vajta paryant ughada aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan aahe, 75 rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 75, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have. 930 rupees", "answer": {"in_stock": true, "price_inr": 930, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this?. It's in stock. Price is 260", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "No, we don't have it. I can give Jupiter clutch plate instead. Home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Jupiter clutch plate"}}
{"product": "bike headlight bulb", "said": "aamhi te thevat nahi, delivery deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, sampla, tyachya aivaji 3 inch hinge deu shakto, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3 inch hinge"}}
{"product": "wall putty 20 kg", "said": "haan hai, rate 710 hai", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahi aahe, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, 4 hi bache hain, 110 rupaye ka hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bola, udya yeil, aaj nahi, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan, stock mein hai, 370 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello, kaun?, nahi hai, delivery ho jayegi", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 240 ka padega", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए। हाँ जी, मिल जाएगा। 230 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, available hai, rate 1000 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hai bhai, 16 piece pade hain, ₹360", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Not sure, I'll have to check the godown. We are open till 7 pm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी, मिल जाएगा। 70 रुपये का है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aamhi te thevat nahi, tyachya aivaji 12V 5Ah battery deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 5Ah battery"}}
{"product": "4 socket extension board", "said": "haan bhai, hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। हाँ जी, मिल जाएगा। रेट 140 है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, available. Home delivery is available. Come before 7", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, udya yeil, aaj nahi, tyachya aivaji 12W LED bulb deu shakto, delivery deto, 9 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "9W LED bulb", "said": "ho aahe, 70 rupaye honar, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe aahe, rate 2100 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "available hai, 330 ka padega, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I can give 3 inch brush instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch brush"}}
{"product": "water pump seal kit", "said": "haan ji, mil jayega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए। नहीं है। घर पर भेज देंगे। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ji boliye, nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Out of stock. I can give local brake shoe instead. We are open till 10 pm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "50 mm padlock", "said": "woh hum nahi rakhte, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "It's in stock. Price is 400. We don't deliver", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नाही आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा। 60 लगेगा", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "We don't keep that. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई। 4 पीस पड़े हैं। रेट सौ है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, nahi aahe, tyachya aivaji LED headlight bulb deu shakto, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED headlight bulb"}}
{"product": "Activa air filter", "said": "malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "संपलं. 40 mm padlock आहे, चालेल का?. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm padlock"}}
{"product": "bike headlight bulb", "said": "Yes, we have it. That will be Rs 220. Pickup only", "answer": {"in_stock": true, "price_inr": 220, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, hai bhai, ₹270, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho aahe, 110 la padel", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji boliye, haan ji, mil jayega, 14 piece pade hain, 80 rupaye final", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "nahi aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे ना. 3 पीस आहेत. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not today, it comes tomorrow. I can give Jupiter clutch plate instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "4 inch door hinge", "said": "available hai, 80 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji haan, rakha hai, 3 hi bache hain, 320 lagega", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock madhe aahe, rate 330 aahe, porga pathavto", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही आहे. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bola, haan aahe, 80 la padel, yeun nyava lagel, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes, available. We have 18 pieces. It's 780 rupees", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes sir, we have. 10 rupees", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "nahi hai, local spark plug hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, 10W-30 engine oil 900 ml rakha hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "10W-30 engine oil 900 ml"}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ बोलिए। नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो, कोण?. संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Yes, available. It's 1450 rupees. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes sir, we have. Only 19 left. 170 rupees", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello, kon?, ho, milel, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan bola, ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए। है भाई। 16 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "malik abhi nahi hain, baad mein phone karo, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 35 rupaye honar", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, available. Price is 100. Home delivery is available. We are open till 8 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला. आम्ही ते ठेवत नाही. 12V 9Ah battery आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 9Ah battery"}}
{"product": "16A MCB", "said": "आहे ना. 180 रुपये. येऊन न्यावं लागेल. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel, 600 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we have it. 280 rupees. We are open till 7 pm", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, mil jayega, dhai sau ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हॅलो, कोण?. आहे ना. 5 पीस आहेत. रेट 200 आहे. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "रुकिए, देखकर बताता हूँ। दुकान 10 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, tell me. Yes, available. That will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो, कौन?। हाँ, है। 330 रुपये का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai bhai, 340 ka padega", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I can give mechanical seal instead. We don't deliver. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "1 inch PVC elbow", "said": "woh hum nahi rakhte, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan ji, mil jayega, 13 piece pade hain, ₹1100, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Hello?. Yes, available. We can deliver", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hello, kaun?, hai bhai, rate 340 hai", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "sampla, 60 mm padlock aahe, chalel ka?, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "9W LED bulb", "said": "नहीं है। उसकी जगह 7W LED bulb दे सकते हैं। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "7W LED bulb"}}
{"product": "2 inch paint brush", "said": "haan bolo, nahi milega bhai, home delivery hai, dukaan 7 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "haan, stock mein hai, 17 piece pade hain, ₹330", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ, है। चालीस लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
# Real Hindi answers from Amazon xPQA (CDLA-Sharing-1.0; see data/real/NOTICE-xpqa.md)
REAL_XPQA = [json.loads(line) for line in r'''
{"product": "pTron Bassbuds Vista true wireless earbuds", "said": "यह अभी उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Vivo V21 5G (Sunset Dazzle, 8GB RAM, 128GB)", "said": "नहीं। यह मोबाईल अभी नहीं मिल रहा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "यह उत्पाद स्टॉक में है। कार्ट में डालके ले सकते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "अभी यह फोन स्टॉक में है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "हाँ। यह झारखंड में उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Paurush Jeevan + Perfect capsule combo pack", "said": "पौरुष जीवन और परफेक्ट केपसुल का कोम्बो पॅक नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ashwagandha, safed musli, shatavar, kaunch beej mixed pack", "said": "नहीं। अश्वगंधा,सफेद मूसली, सतावर, कौंच बीज, इन सबको मिलाकर मिला मिलाया आनुपातिक पैक नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wecool Moonwalk Mini earbuds charging case (spare)", "said": "नहीं। चार्जिंग केस अलग से नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "इसका ब्लेड अलग से नहीं मिलता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Ambrose ceiling fan, 900 mm", "said": "नहीं। 900 mm में उपलब्ध नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "XS and OS women's nightwear set, 40 inch size", "said": "नहीं। इसमें 40 इंच का नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं, ब्लैक कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Rehmans Mumbai model wooden sofa set, other colour", "said": "नहीं, इसमें दूसरा कलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं, रेड कलर में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Oomph georgette salwar suit material, stitched", "said": "नहीं। यह कपडा सिला हुआ नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Anni Designer crepe dress material, stitched", "said": "नहीं। यह सलवार सूट सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kundan Sulz suit fabric, stitched", "said": "नहीं। यह सिला हुआ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "नहीं। यह 19 इंच की नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebster LED monitor, 19 inch", "said": "हाँ। यह 19 इंच की है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "हाँ। यह सैमसंग a12 6gb 128gb का है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells Pacer 1200 mm ceiling fan, 5 star rated", "said": "नहीं। यह 5 स्टार नहीं है, इस फैन की कोई स्टार रेटिंग नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ फ्री साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"L\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "L साइज़"}}
{"product": "Lujoso women's rayon kurti (6XL)", "said": "नहीं। यह सिर्फ \"M\", \"XL\" और \"XXL\" साइज़ में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M, XL और XXL साइज़"}}
{"product": "Human hair side patch, set of 2 (black)", "said": "नहीं। यह सिर्फ डार्क ब्राउन(गहरा भूरा) रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "डार्क ब्राउन"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "ग्रे रंग"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ चारकोल ब्लू में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "चारकोल ब्लू"}}
{"product": "Damensch modal trunks, pack of 2 (yellow)", "said": "नहीं। यह पीले रंग में नहीं मिलेगा, यह सिर्फ सिल्वर और ग्रे में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिल्वर और ग्रे"}}
{"product": "Fire-Boltt Ninja Calling smartwatch (red)", "said": "नहीं। यह मेटल ग्रे रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "मेटल ग्रे रंग"}}
{"product": "Pare leather jacket, colour other than black", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "iBell Eiffel pedestal fan, other colour", "said": "नहीं। यह सिर्फ काले रंग में मिलेगा जिसपे हल्का कॉपर रंग होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Autonest bike side luggage box, Splendor Plus (silver)", "said": "नहीं। यह काले रंग में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग"}}
{"product": "Tecno Spark 6 Air (black, 3GB RAM, 64GB)", "said": "नहीं। 3gb रैम और 64gb स्टोरेज के साथ नीले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "नीले रंग"}}
{"product": "SDS Fashion boys' t-shirt, other colour, 11-12 years", "said": "नहीं। यह 13-14 के बच्चों के लिए काले रंग में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "काले रंग, 13-14"}}
{"product": "Oomph georgette salwar suit material (XXL)", "said": "नहीं। यह फ्री साइज़ का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फ्री साइज़"}}
{"product": "Nir Fashion net lehenga choli (3XL)", "said": "नहीं। यह ड्रेस 2xl है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2xl"}}
{"product": "Samsung Galaxy A12 (6GB RAM, 128GB)", "said": "नहीं। यह सैमसंग a12 4gb 64gb का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सैमसंग a12 4gb 64gb"}}
{"product": "L&T three phase energy meter", "said": "यह एल&टी का सिंगल फेज मीटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिंगल फेज मीटर"}}
{"product": "Shree multi-output transformer, copper winding", "said": "नहीं। यह एल्युमिनियम का ट्रांसफार्मर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एल्युमिनियम का ट्रांसफार्मर"}}
{"product": "Activa 3 litre instant gas geyser", "said": "नहीं। यह इलेक्ट्रिक गाइज़र है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इलेक्ट्रिक गाइज़र"}}
{"product": "Microtek MTK3012SMU solar inverter", "said": "नहीं। यह इनवर्टर नहीं है, सिर्फ सोलर चार्ज कन्ट्रोलर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सोलर चार्ज कन्ट्रोलर"}}
{"product": "Fire-Boltt 360 Pro smart watch, metal body", "said": "नहीं। यह मेटल बॉडी नहीं है, सिलिकॉन बॉडी है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सिलिकॉन बॉडी"}}
{"product": "Women's regular kurta (black & grey), cotton", "said": "नहीं। यह कॉटन नहीं है, यह रेयॉन का कपडा है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "रेयॉन का कपडा"}}
{"product": "Leather Retail men's biker jacket, pure leather", "said": "नहीं। यह प्यूर लेदर नहीं है, फॉक्स लेदर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फॉक्स लेदर"}}
{"product": "Jaspo heavy duty cricket bat, wooden", "said": "नहीं। यह लकड़ी का नहीं प्लास्टिक का है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "प्लास्टिक"}}
{"product": "HP all-in-one desktop PC with Intel i5", "said": "नहीं। इसमें एएमडी राइज़न है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एएमडी राइज़न"}}
{"product": "Epson all-in-one laser printer", "said": "नहीं। यह इंक प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "इंक प्रिंटर"}}
{"product": "Webby wooden India political map puzzle, Hindi", "said": "नहीं। यह अंग्रेजी में लिखा हुआ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अंग्रेजी"}}
{"product": "Faber 60 cm kitchen chimney, island model", "said": "नहीं। यह वाल माउंट है। आइलैंड के लिए दुकान जाने की जरूरत है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "वाल माउंट"}}
{"product": "Niyama Bluetooth thermal printer, A4 size", "said": "नहीं। यह सिर्फ 2 इंच प्रिंटर है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2 इंच प्रिंटर"}}
{"product": "Dot bathroom tiles cleaner, 5 litre jar", "said": "नहीं। यह 500 एम एल का जार है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 एम एल का जार"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। यह 27*24*4 इंच का पेड़ है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "27*24*4 इंच"}}
{"product": "Lilycool honeycomb cooling pad, 39 x 39 x 4 inch", "said": "नहीं। 36\" (h) x 24\" (w) x 4\" का पैड मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "36\" x 24\" x 4\" का पैड"}}
{"product": "Bru instant coffee, pack bigger than 200 g", "said": "नहीं। ब्रू कॉफी 200 gm के पैक में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "200 gm के पैक"}}
{"product": "Katra Lysorus virucide, pack smaller than 50 ml", "said": "नहीं। यह सिर्फ 50 ml के पैक में मिलता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "50 ml के पैक"}}
{"product": "Baidyanath Nagpur Swarna Bhasma, 500 g", "said": "नहीं। यह 125mg में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "125mg"}}
{"product": "Zandu Kesari Jivan, 2 litre pack", "said": "नहीं। यह 900g के पैक में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "900g के पैक"}}
{"product": "Herbalife Shakemate protein shake, 1 kg pack", "said": "यह 600 g का पैक है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "600 g का पैक"}}
{"product": "Easyshoppingbazaar bird net, 25 x 25 ft", "said": "नहीं। यह आईटम 10 x 6 foot (60 sq ft) में मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "10 x 6 foot"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 32*10 में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "32*10"}}
{"product": "Shade net 50% UV stabilised, 20 x 20 metre", "said": "नहीं। यह 3*10m में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3*10m"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 12 सेंटीमीटर लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12 सेंटीमीटर लम्बाई"}}
{"product": "SBJ girls' ruffle maxi palazzo, 42 length", "said": "नहीं। यह 39 लम्बाई में उपलब्ध है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "39 लम्बाई"}}
{"product": "Car mat for new Tata Safari 7-seater", "said": "यह सफारी 6 सीटर के लिए है और इसका दाम 1420 रुपये है डिस्काउंट के बाद।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सफारी 6 सीटर"}}
{"product": "Renmax walkie talkie, single piece", "said": "नहीं। इस पैक में दो मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "पैक में दो"}}
{"product": "Zoro waist belt, pack of 3", "said": "इस कॉम्बो में २ पीस मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "२ पीस"}}
{"product": "Queenzomy bop bag toy, set of 3", "said": "नहीं। इसमें सिर्फ एक पीस आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Status car door guard, set of 4", "said": "इसमें केवल एक पीस आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पीस"}}
{"product": "Flamingo calf support, pair (both legs)", "said": "इसमें सिर्फ एक पैर की आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एक पैर"}}
{"product": "Kozdiko car foot mat for Renault Kwid Climber", "said": "नहीं। यह ऐल्टो के लिए है, क्विड पे नहीं लगेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Raya keyboard skin for 15.6 inch laptop", "said": "नहीं। यह 14 इंच लैपटाप के लिये उपयुक्त है, 15.6 इंच लैपटाप के लिये उपयुक्त नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Casotec silicone back cover for Tecno iA5", "said": "नहीं। यह टेक्नो केमन i ट्विन के लिए बना है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Eltron H4 headlight bulb for Bajaj Pulsar 125", "said": "नहीं। यह बजाज पल्सर 125 के हेडलाइट में फिट नहीं होगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Live Tech 12A toner cartridge for HP 1020 Plus", "said": "नहीं। इसे एच पी 1020 में लगा सकते है पर एच पी 1020 प्लस में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "AOW brake and clutch levers for Honda Shine SP 125", "said": "नहीं। यह होंडा शाइन एसपी 125 में फिट नहीं होता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JK Tyre Blaze BF32 front tyre for Honda 110cc", "said": "नहीं। यह होंडा 110cc में नहीं चलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Autofurnish car cover for Swift Dzire 2015", "said": "नहीं। यह किया सॉनेट के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kingsway car body cover for Alto LX 2008", "said": "नहीं। यह कवर शेव्रोले क्रूज़ के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Shopoflux silicone key cover for Hyundai Eon", "said": "नहीं। यह हुंडई इयान के लिए परफेक्ट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Front touch glass for Huawei Y9 2019", "said": "नहीं। यह टच स्क्रीन ग्लास हुआवे y9 प्राइम 2019 में यूज़ हो सकता है, y9 2019 में नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Loom Solar 20A charge controller for LiFePO4 battery", "said": "नहीं। यह सिर्फ लेड बैटरी के लिए है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Micromax X412 phone with headphones", "said": "इसमें एक चार्जर साथ में उपलब्ध है पर हेडफोन साथ में नहीं दिया जाता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Tender Care plastic almirah with cover", "said": "नहीं। इसमें सिर्फ अलमीरा मिलेगा, कवर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "अलमीरा"}}
{"product": "Tender Care plastic almirah with cover", "said": "हाँ। इस अलमीरा का एक रजाई जैसा कवर मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Blaxstoc Bluetooth earphones with mobile stand", "said": "हाँ। इसके साथ मोबाइल स्टैंड मिलेगा।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Microtek solar PCU 1235 with solar panel", "said": "इसमें सौर पैनल शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "हाँ। इसके साथ 2aa बैटरी मिलेगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ionix kitchen weighing scale with batteries", "said": "नहीं। इसके साथ बैटरी नहीं मिलेगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Omo Bikes Panda 20T kids bicycle with carrier", "said": "इस में कैरियर नहीं होगा, चेन कवर पहले से ही रहता है क्रैंक वाली साइड पर।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar soundbar with mic", "said": "नहीं। मैक साथ में उपलब्ध नहीं है। fm नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Kuki georgette saree with blouse piece", "said": "हाँ। इसके साथ ब्लाउज पीस है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Usha Aquerra 25 L water heater with remote", "said": "नहीं। इसके साथ रिमोट नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Monkish Apple Watch case with band", "said": "इसमें केवल घड़ी का केस मिलेगा स्क्रीन प्रोटेक्टर के साथ पर घडी या बेंड नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "घड़ी का केस"}}
{"product": "mStick silicone AirPods cover with AirPods", "said": "नहीं। आइटम में केवल एयरपोड कवर है और एयरपोड शामिल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "एयरपोड कवर"}}
{"product": "Symphony Storm 100i tower air cooler", "said": "यह सिर्फ कूलर का कवर है, इसमें कूलर नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ayushka Crafts tent ceiling with side walls", "said": "नहीं। साइडर साथ में नहीं मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें सिर्फ बीन बैग का कवर मिलेगा, बीन्स नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "बीन बैग का कवर"}}
{"product": "Mollismoons fur bean bag XXXL with beans", "said": "नहीं। इसमें बीन्स साथ में नहीं मिलेंगे।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pro-Ethic men's cotton kurta with pyjama", "said": "इसमें सिर्फ एक कुर्ता मिलेगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "कुर्ता"}}
{"product": "Rain gun set with rain gun, 4-leg stand and riser", "said": "हाँ। यह सेट रेन गन के साथ आती है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "JBL GT-X1500THI subwoofer with amplifier", "said": "नहीं। इसमें सिर्फ सबवूफर आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "सबवूफर"}}
{"product": "Hectare mango fruit picker with pole", "said": "नहीं। इसके साथ डंडा नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajab Gajab solar zatka machine 12 kV with battery", "said": "नहीं। इसके साथ बैटरी नहीं आएगी।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Coirfit 3-seater sofa cum bed with stand", "said": "नहीं। स्टैंड साथ में नहीं आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Smartcam bulb CCTV camera with memory card", "said": "नहीं। इसमें मेमोरी कार्ड साथ में नहीं आती है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mi Pad 5 with pen", "said": "पेन साथ में नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G with 18 W adapter in box", "said": "नहीं। बॉक्स में 18 वॉट का एडाप्टर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Volo all-in-one makeup kit with blusher", "said": "इसमें ब्लशर नहीं आता है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme Narzo 50A with charger", "said": "हाँ। इस मोबाईल के साथ चार्जर आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with aux cable", "said": "हाँ। इसके साथ ऑक्स केबल आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Bang Pro headphones with charging and aux cable", "said": "हाँ। इसके साथ चार्जर केबल और ऑक्स केबल आएंगी।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Noise ColorFit Ultra smart watch with charger", "said": "हाँ। इसमें चार्जिंग डॉक साथ में आता है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Globe Dolphin silencer for RE Classic 350 with mounting kit", "said": "हाँ। इस साइलेंसर को लगाने की पत्ती और बोल्ट साथ में आते है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Lovely 12V solar pedestal fan with battery and solar panel", "said": "नहीं। ये बैटरी और सोलर पेनल के साथ नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dell Inspiron N5010 65W adapter with power cable", "said": "नहीं। इसमें एडेप्टर के साथ पावर केबल नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Wacom One graphic tablet with pen battery", "said": "नहीं। इस पेन टेब के साथ बैटरी नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Sahni single folding bed with mattress", "said": "नहीं, इसके साथ बिस्तर नहीं आता।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Docoss rechargeable head torch with charger", "said": "नहीं। इसमें चार्जर नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "TVS Eurogrip 2.75-18 rear tyre with tube", "said": "नहीं। टायर के साथ में ट्यूब नहीं है।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Veera Paridhaan kurta pyjama", "said": "कुर्ता पजामा आइटम की डिलीवरी जल्दी हो सकती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Samsung 6.5 kg top load washing machine", "said": "हाँ। गडचिरोली में डेलीवरी होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Bharat Darshan wooden agarbatti stand", "said": "हाँ। इस कंपनी द्वारा विदेश के मित्र को उत्पाद भेज सकते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "हाँ। 499 तक के प्रोडक्ट पे कक्ष ऑन डेलीवेरी मिल सकता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Mecstroke weed cutter wheat-cutting attachment", "said": "गेहूं काटनेवाला मशीन 15 दिन बाद भेज जाता है जिसे सेट कर सकते हो।", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Senapati men's cotton socks (pack of 12)", "said": "₹359 में 12 जोड़ी मोज़े है।", "answer": {"in_stock": true, "price_inr": 359, "delivery": null, "alternative": null}}
{"product": "Shade net 50% UV stabilised garden netting", "said": "रू 259 में 5ft x32ft है।", "answer": {"in_stock": true, "price_inr": 259, "delivery": null, "alternative": null}}
{"product": "Apis Himalaya honey, 1 litre", "said": "312/- में 35.27 औंस यानी 1 लीटर मिलेगा।", "answer": {"in_stock": true, "price_inr": 312, "delivery": null, "alternative": null}}
{"product": "iQOO Z6 5G (Dynamo Black, 6GB RAM, 128GB)", "said": "812 रुपये किस्त में फोन उपलब्ध है।", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Zebronics Zeb-Jukebar 3900 soundbar", "said": "इस उत्पाद पे लिए हुए दिन से १ साल तक की मैनुफैक्चरर वारंटी मिलती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Softspun microfiber cleaning cloths, 24 pcs", "said": "इस उत्पाद में 24 टावल पीसस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Himalaya Total Care baby pants diapers, medium", "said": "एक बॉक्स मे ७८ पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "NSSP flat lanyards with J-hook (pack of 50)", "said": "इसमें ५० पीस है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Romsons 6-layer disposable face mask (pack of 50)", "said": "एक पैक में 25 पीस आते है और ऐसा 2 पैक आते है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "VBCFT GK50 battery for Moto E3", "said": "नहीं। इसमें सिर्फ तीन महीने की वारंटी मिलेगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Godrej 1.5 ton 3 star inverter split AC", "said": "एसी का इन्स्टलैशन गोदरेज वाले करेंगे और इसका इन्स्टलैशन चार्ज लगेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi Note 11 Pro (Star Blue, 6GB RAM, 128GB)", "said": "कम से कम किस्त 2222 रुपये की होगी।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "3 महीने का 2327 रुपये हर महीना और 6 महीने का 1163 रुपये हर महीने का किस्त होगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ajjas hidden GPS tracker for bike", "said": "महिने का 200 रुपये का डाटा लागत है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Cera angle cock with wall flange", "said": "एक पैकेट में 1 एंगल वाल आता है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "नहीं। पुरानी बैटरी एक्सचेंज नहीं होती है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Canon GI-790 ink bottle (black)", "said": "नहीं। 4-5शीशी एक साथ मंगाने पर डिस्काउंट नहीं मिलेगा।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "BigSavings 20 LED solar motion sensor light", "said": "Cash on delivery mil sakta hai iske liye aapko 499 Tak ka product karna padega lekin yah Photo bahut kharab hai breakup bahut bekar hai", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "GMDT 10 kVA generator (three phase)", "said": "yes it is available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Redmi 10 Prime (Bifrost Blue, 4GB RAM, 64GB)", "said": "In stock.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Webby beauty set for girls (pink)", "said": "In stock. Add to cart.", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Syska HT200U trimmer blade (spare)", "said": "Extra Accessories is not available. You want extra accessories so you can call customer care.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Mecstroke weed cutter wheat and grass cutting blades", "said": "HELLO , WE WILL SEND THE WHEAT AND GRASS CUTTING BLADES TO YOU 15 DAYS AFTER THE MACHINES IS DELIVERED.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Realme C11 (2021) (Cool Grey, 2GB RAM, 32GB)", "said": "Delivery charge is free and EMI depend on the offer available at the time of purchase with the debit or credit card.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Clovia cotton non-padded wirefree bra", "said": "Delivered on time only.", "answer": {"in_stock": null, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "HBMallIndia anti-burst gym ball", "said": "10CM ka fark he sizes me. 65cm, 75cm aur 85cm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Genus Invoshakti solar tubular battery 150 Ah", "said": "Hello Sir, We are currently not offering exchange services online, we would request you to get best buyback prices from our authorized dealers in your nearby area.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
TRAIN_V1 = [json.loads(line) for line in r'''
{"product": "Splendor brake shoe", "said": "haan bolo, hai bhai, home delivery hai, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, haan, stock mein hai, 220 rupaye ka hai, home delivery hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "Yes, available. It's 80 rupees", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes sir, we have. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "वो हम नहीं रखते। उसकी जगह 3.15 mfd capacitor दे सकते हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3.15 mfd capacitor"}}
{"product": "silicone sealant", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ जी, मिल जाएगा। 160 लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 160, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "aahe na, delivery deto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan bola, aahe na, 10 piece aahet, 80 rupaye honar, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Not sure, I'll have to check the godown. Come before 10", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 16 piece aahet, vees rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 20, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, ho, milel, pannas rupaye", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. No, we don't have it. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "khatam ho gaya, uski jagah 7W LED bulb de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "7W LED bulb"}}
{"product": "4 inch door hinge", "said": "स्टॉक में है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, didshe rupaye", "answer": {"in_stock": true, "price_inr": 150, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 110 lagega", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "हाँ, है। 160 रुपये का है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "Not today, it comes tomorrow. We don't deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "aahe na, 11 piece aahet, rate 390 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, available. That will be Rs 160", "answer": {"in_stock": true, "price_inr": 160, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ, है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan, stock mein hai, 1600 rupaye ka hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "16A MCB", "said": "available hai, rate 310 hai", "answer": {"in_stock": true, "price_inr": 310, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "haan bhai, hai bhai, 15 piece pade hain, rate 360 hai", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हां बोला. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. हो, आहे. 950 ला पडेल. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 950, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan ji, mil jayega, rate 380 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 380, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "हाँ बोलिए। वो हम नहीं रखते। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan, stock mein hai, 5 piece pade hain, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aahe na, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ji boliye, nahi hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, tell me. Yes, we have it", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji boliye, available hai, 18 hi bache hain, ₹15, dukaan 8 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, aahe na, 14 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "haan bhai, abhi nahi hai, parso aayega, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, we have it. Price is 35. We can deliver", "answer": {"in_stock": true, "price_inr": 35, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "stock madhe aahe, 50 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bola, mahit nahi, godown madhe baghava lagel", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, haan, stock mein hai, 160 lagega, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 160, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan bhai, ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "थांबा, बघून सांगतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "Hello, who is this?. Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "available hai, dhai sau rupaye final, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "available hai, 2 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "स्टॉक में है। रेट 310 है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 310, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. पन्नास ला पडेल. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "ji haan, rakha hai, 20 lagega, delivery ho jayegi, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 20, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "hello, kaun?, hai bhai, 410 rupaye final", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, we have it. Price is 1150", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "है भाई। 1100 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, ghari pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "मालक नाहीत, नंतर फोन करा. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। ख़त्म हो गया। घर पर भेज देंगे। दुकान 8 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ji haan, rakha hai, 130 rupaye final, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "स्टॉकमध्ये आहे. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bola, aahe na, rate 1600 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 1600, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "hello, kaun?, thodi der baad call karo, dukaan 8 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bhai, abhi stock mein nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aamhi te thevat nahi, tyachya aivaji CPVC elbow deu shakto, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "CPVC elbow"}}
{"product": "12V 7Ah battery", "said": "आहे ना. 9 पीस आहेत. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan, stock mein hai, 2150 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 2150, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi nahi hai, parso aayega, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "आम्ही ते ठेवत नाही. local putty आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local putty"}}
{"product": "Activa air filter", "said": "है भाई। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan ji, mil jayega", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "available hai, 15 piece pade hain, 35 ka padega", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give CPVC elbow instead. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "silicone sealant", "said": "ji boliye, haan ji, mil jayega, 180 rupaye final, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "हैलो, कौन?। रुकिए, देखकर बताता हूँ। दुकान 8 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हॅलो, कोण?. नाही आहे. Pulsar 180 spark plug आहे, चालेल का?. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Pulsar 180 spark plug"}}
{"product": "Activa air filter", "said": "nahi hai, ladka bhej deta hun, dukaan 10 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan aahe, 7 piece aahet, 380 rupaye honar", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ho, milel, 1750 la padel, ghari pathavto", "answer": {"in_stock": true, "price_inr": 1750, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai, haan hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, uski jagah local clutch plate de sakte hain, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local clutch plate"}}
{"product": "10W-30 engine oil 1 L", "said": "Yes, available", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 5 hi bache hain, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "ji boliye, available hai, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "haan hai, 6 hi bache hain, ₹140", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 60 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हां बोला. आहे ना. 90 रुपये", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan hai, 830 lagega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": 830, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "ji boliye, pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, thread seal tape hai, chalega?, delivery nahi karte, aake le jao, dukaan 9 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "thread seal tape"}}
{"product": "50 mm padlock", "said": "ji boliye, haan, stock mein hai, 17 hi bache hain, 150 ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 150, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 14 piece aahet, 50 rupaye, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 50, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "abhi nahi hai, parso aayega, 10 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "haan bolo, hai bhai, 400 rupaye final", "answer": {"in_stock": true, "price_inr": 400, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हो, आहे. 1450 रुपये", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bhai, available hai, 610 lagega", "answer": {"in_stock": true, "price_inr": 610, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 5 पीस पड़े हैं। 1600 रुपये का है। घर पर भेज देंगे। दुकान 7 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 1600, "delivery": true, "alternative": null}}
{"product": "water pump seal kit", "said": "It's in stock. We have 14 pieces. Price is 180. I'll send it with our boy. Come before 10", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "abhi stock mein nahi hai, uski jagah CPVC elbow de sakte hain, ladka bhej deta hun, 9 baje ke pehle aana", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "CPVC elbow"}}
{"product": "Activa air filter", "said": "नहीं है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "है भाई", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "आहे ना. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "सध्या नाही, उद्या येईल. paint roller आहे, चालेल का?. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "2 inch paint brush", "said": "abhi nahi hai, parso aayega, paint roller hai, chalega?, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "paint roller"}}
{"product": "fan capacitor 2.5 mfd", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "hello, kon?, sampla, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "abhi stock mein nahi hai, 7W LED bulb hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "7W LED bulb"}}
{"product": "12V 7Ah battery", "said": "It's in stock. It's 1050 rupees. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 1050, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "aahe na, shambhar rupaye", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "We don't keep that", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "ख़त्म हो गया। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "abhi stock mein nahi hai, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji boliye, abhi nahi hai, parso aayega, uski jagah 60 mm padlock de sakte hain, home delivery hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "bike headlight bulb", "said": "Hello?. It's in stock. Home delivery is available", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi milega bhai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, 130 rupaye", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "stock madhe aahe, 410 la padel", "answer": {"in_stock": true, "price_inr": 410, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "malik abhi nahi hain, baad mein phone karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "haan bolo, ji haan, rakha hai, 330 ka padega", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "haan aahe, 19 piece aahet", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "Yes, we have it. Price is 290. Pickup only", "answer": {"in_stock": true, "price_inr": 290, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bolo, haan ji, mil jayega, 120 lagega", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "Hello, who is this?. It's in stock. That will be Rs 100. We are open till 10 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "It's in stock. 70 rupees. I'll send it with our boy. Come before 9", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 rupaye honar", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "haan hai, 30 lagega, ghar pe bhej denge, 7 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है। paint roller है, चलेगा?। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "paint roller"}}
{"product": "10W-30 engine oil 1 L", "said": "haan ji, mil jayega, rate 380 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 380, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Yes, we have it. That will be Rs 1800. We don't deliver", "answer": {"in_stock": true, "price_inr": 1800, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हो, आहे. 8 पीस आहेत", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 210 rupaye ka hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 210, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, rate 1300 hai", "answer": {"in_stock": true, "price_inr": 1300, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bola, thamba, baghun sangto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "nahi milega bhai, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "sadhya nahi, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "Yes, we have it. That will be Rs 240", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "aamhi te thevat nahi, tyachya aivaji wall putty 40 kg deu shakto, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "wall putty 40 kg"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हाँ बोलिए। स्टॉक में है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "ji haan, rakha hai, 75 rupaye ka hai", "answer": {"in_stock": true, "price_inr": 75, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan bhai, nahi milega bhai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, hai bhai, ₹1500", "answer": {"in_stock": true, "price_inr": 1500, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. हो, आहे. 13 पीस आहेत. 120 रुपये. घरपोच देतो", "answer": {"in_stock": true, "price_inr": 120, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "haan, stock mein hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, tell me. Not today, it comes tomorrow. I can give 3/4 inch elbow instead. Pickup only", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3/4 inch elbow"}}
{"product": "9W LED bulb", "said": "आहे ना. 3 पीस आहेत. रेट 140 आहे", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 30 rupaye honar, delivery deto", "answer": {"in_stock": true, "price_inr": 30, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan bola, ho aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "रुकिए, देखकर बताता हूँ", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan ji, mil jayega, 80 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "ji boliye, hai bhai, rate pachaas hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 50, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "नहीं है। उसकी जगह 12V 9Ah battery दे सकते हैं। घर पर भेज देंगे", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12V 9Ah battery"}}
{"product": "teflon tape", "said": "ji boliye, haan ji, mil jayega, rate 30 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "bola, thamba, baghun sangto, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Not today, it comes tomorrow", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "woh hum nahi rakhte, LED headlight bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "LED headlight bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan ji, mil jayega, 3 hi bache hain, ₹80, home delivery hai", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "abhi stock mein nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan bolo, haan ji, mil jayega, rate 50 hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 50, "delivery": null, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "hai bhai, 14 piece pade hain, delivery ho jayegi, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "It's in stock. That will be Rs 1700", "answer": {"in_stock": true, "price_inr": 1700, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "नाही आहे. त्याच्याऐवजी 3.15 mfd capacitor देऊ शकतो. घरी पाठवतो", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "3.15 mfd capacitor"}}
{"product": "Fevicol SH 1 kg", "said": "bola, ho aahe, 420 rupaye honar", "answer": {"in_stock": true, "price_inr": 420, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "स्टॉक में है। रेट 2200 है। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 2200, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hai bhai, 7 hi bache hain, ₹120, dukaan 10 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 120, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "pata nahi, godown mein dekhna padega", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "ho, milel, 9 ch urle aahet, 70 rupaye honar", "answer": {"in_stock": true, "price_inr": 70, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हाँ जी, मिल जाएगा। 1150 लगेगा", "answer": {"in_stock": true, "price_inr": 1150, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "हाँ बोलिए। ख़त्म हो गया। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "है भाई। 230 लगेगा। दुकान 10 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 230, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "ho, milel, 270 rupaye honar", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "stock madhe aahe, adichshe rupaye honar, ghari pathavto", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "ho, milel, rate 90 aahe, ghari pathavto, 10 chya aat ya", "answer": {"in_stock": true, "price_inr": 90, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "nahi milega bhai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "ji haan, rakha hai, 5 piece pade hain, 220 lagega", "answer": {"in_stock": true, "price_inr": 220, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "silicone sealant", "said": "हाँ, है। 13 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "woh hum nahi rakhte, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan bolo, hai bhai, 1450 rupaye ka hai, dukaan 9 baje tak khuli hai", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, we have it. Price is 45. Pickup only", "answer": {"in_stock": true, "price_inr": 45, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "मालिक नहीं हैं, बाद में फ़ोन कीजिए", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan bolo, nahi hai, uski jagah 60 mm padlock de sakte hain", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "60 mm padlock"}}
{"product": "10 mm nut bolt (pack of 10)", "said": "khatam ho gaya, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "हैलो, कौन?। हाँ, है। 20 पीस पड़े हैं। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Hello, who is this?. Owner is not here, call later", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "Yes sir, we have. We have 4 pieces. That will be Rs 90. Pickup only", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "aamhi te thevat nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "हॅलो, कोण?. आहे ना. 80 रुपये. घरी पाठवतो. दुकान 8 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 80, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "haan ji, mil jayega, 730 ka padega", "answer": {"in_stock": true, "price_inr": 730, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "सध्या नाही, उद्या येईल. डिलिव्हरी नाही, दुकानात येऊन घ्या", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "ji haan, rakha hai, rate 240 hai, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हाँ, है। 120 लगेगा। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "available hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "nahi milega bhai, halogen bulb hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "halogen bulb"}}
{"product": "1 inch PVC elbow", "said": "हां बोला. स्टॉकमध्ये आहे. तीस रुपये", "answer": {"in_stock": true, "price_inr": 30, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "Yes, available. 65 rupees", "answer": {"in_stock": true, "price_inr": 65, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "It's in stock. No delivery, you'll have to come to the shop", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji haan, rakha hai, 100 rupaye final, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. It's in stock. We have 9 pieces. It's 220 rupees. We can deliver", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "nahi milega bhai, local brake shoe hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "4 socket extension board", "said": "hai bhai, 520 ka padega", "answer": {"in_stock": true, "price_inr": 520, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "हॅलो, कोण?. थांबा, बघून सांगतो", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "haan aahe, 10 rupaye, yeun nyava lagel, 9 chya aat ya", "answer": {"in_stock": true, "price_inr": 10, "delivery": false, "alternative": null}}
{"product": "4 inch door hinge", "said": "pata nahi, godown mein dekhna padega, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Hello?. No, we don't have it. We have local seal kit, will that work?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Activa clutch plate", "said": "malik abhi nahi hain, baad mein phone karo, 9 baje ke pehle aana", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "malak nahit, nantar phone kara, 9 chya aat ya", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "ji haan, rakha hai, 1500 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 1500, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Yes, tell me. We don't keep that. I can give local brake shoe instead. Home delivery is available. Come before 8", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "local brake shoe"}}
{"product": "bike headlight bulb", "said": "hai bhai, 7 piece pade hain, home delivery hai", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Yes, tell me. Yes sir, we have. Price is 350", "answer": {"in_stock": true, "price_inr": 350, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Sorry, finished. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "khatam ho gaya, 2.25 mfd capacitor rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "2.25 mfd capacitor"}}
{"product": "Pulsar 150 spark plug", "said": "आम्ही ते ठेवत नाही. local spark plug आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "9W LED bulb", "said": "Hello?. Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan, stock mein hai, ₹110, delivery nahi hai, 8 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 110, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नहीं है", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, haan aahe, 170 rupaye, ghari pathavto, dukan 10 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 170, "delivery": true, "alternative": null}}
{"product": "4 socket extension board", "said": "haan hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "No, we don't have it. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "ho aahe, 120 rupaye, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 120, "delivery": false, "alternative": null}}
{"product": "tank float valve", "said": "available hai, 17 piece pade hain, rate 400 hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hello, who is this?. Sorry, finished. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "haan bola, udya yeil, aaj nahi", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "sampla, 8 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "hello, kaun?, abhi stock mein nahi hai, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "haan bhai, haan, stock mein hai, 6 piece pade hain, 70 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "2 inch paint brush", "said": "hello, kaun?, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "abhi nahi hai, parso aayega, uski jagah 2.25 mfd capacitor de sakte hain, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2.25 mfd capacitor"}}
{"product": "50 mm padlock", "said": "हाँ बोलिए। स्टॉक में है। 2 पीस पड़े हैं", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "thamba, baghun sangto", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "abhi stock mein nahi hai, local seal kit rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local seal kit"}}
{"product": "Pulsar 150 spark plug", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हॅलो, कोण?. स्टॉकमध्ये आहे. 10 पीस आहेत. तीस रुपये. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 30, "delivery": false, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Hello?. Yes, we have it. Only 6 left. It's 1100 rupees", "answer": {"in_stock": true, "price_inr": 1100, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "हाँ, है। 17 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "khatam ho gaya, local air filter hai, chalega?, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "local air filter"}}
{"product": "Pulsar 150 spark plug", "said": "udya yeil, aaj nahi, Pulsar 180 spark plug aahe, chalel ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Pulsar 180 spark plug"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Not today, it comes tomorrow. I'll send it with our boy", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ji haan, rakha hai, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "hello, kaun?, woh hum nahi rakhte, delivery nahi hai, dukaan 8 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "hello, kon?, sampla, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "हाँ, है", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "wall putty 20 kg", "said": "ji haan, rakha hai, 760 rupaye ka hai, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 760, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "haan bhai, khatam ho gaya", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "हॅलो, कोण?. मालक नाहीत, नंतर फोन करा. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "है भाई। रेट 90 है। डिलीवरी नहीं है", "answer": {"in_stock": true, "price_inr": 90, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "ख़त्म हो गया। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Activa clutch plate", "said": "stock madhe aahe", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "हां बोला. नाही आहे. त्याच्याऐवजी 2.5 sq mm wire देऊ शकतो", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "2.5 sq mm wire"}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bhai, haan ji, mil jayega, 70 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "hello, kaun?, nahi hai, 12W LED bulb hai, chalega?, ladka bhej deta hun", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "woh hum nahi rakhte, 1 sq mm wire rakha hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 sq mm wire"}}
{"product": "8 mm drill bit", "said": "haan bhai, available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "sampla, dukan 7 vajta paryant ughada aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "2 inch paint brush", "said": "haan aahe, 75 rupaye, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 75, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have. 930 rupees", "answer": {"in_stock": true, "price_inr": 930, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, delivery nahi hai", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "Hello, who is this?. It's in stock. Price is 260", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "No, we don't have it. I can give Jupiter clutch plate instead. Home delivery is available", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "Jupiter clutch plate"}}
{"product": "bike headlight bulb", "said": "aamhi te thevat nahi, delivery deto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "bola, sampla, tyachya aivaji 3 inch hinge deu shakto, delivery nahi karat", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "3 inch hinge"}}
{"product": "wall putty 20 kg", "said": "haan hai, rate 710 hai", "answer": {"in_stock": true, "price_inr": 710, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, ladka bhej deta hun", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "nahi aahe, porga pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "9W LED bulb", "said": "haan, stock mein hai, 4 hi bache hain, 110 rupaye ka hai, home delivery hai", "answer": {"in_stock": true, "price_inr": 110, "delivery": true, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "bola, udya yeil, aaj nahi, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "haan, stock mein hai, 370 rupaye final, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 370, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "hello, kaun?, nahi hai, delivery ho jayegi", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan hai, 240 ka padega", "answer": {"in_stock": true, "price_inr": 240, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "हाँ बोलिए। हाँ जी, मिल जाएगा। 230 रुपये का है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 230, "delivery": true, "alternative": null}}
{"product": "8 mm drill bit", "said": "sampla", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "ji boliye, available hai, rate 1000 hai, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1000, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hai bhai, 16 piece pade hain, ₹360", "answer": {"in_stock": true, "price_inr": 360, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Not sure, I'll have to check the godown. We are open till 7 pm", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "हाँ जी, मिल जाएगा। 70 रुपये का है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 70, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "aamhi te thevat nahi, tyachya aivaji 12V 5Ah battery deu shakto", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 5Ah battery"}}
{"product": "4 socket extension board", "said": "haan bhai, hai bhai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हैलो, कौन?। हाँ जी, मिल जाएगा। रेट 140 है। घर पर भेज देंगे", "answer": {"in_stock": true, "price_inr": 140, "delivery": true, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "Yes, available. Home delivery is available. Come before 7", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "16A MCB", "said": "ruko, dekh ke batata hun", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "bola, udya yeil, aaj nahi, tyachya aivaji 12W LED bulb deu shakto, delivery deto, 9 chya aat ya", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12W LED bulb"}}
{"product": "9W LED bulb", "said": "ho aahe, 70 rupaye honar, delivery nahi, dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "stock madhe aahe, rate 2100 aahe, delivery nahi karat", "answer": {"in_stock": true, "price_inr": 2100, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "available hai, 330 ka padega, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi stock mein nahi hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "No, we don't have it. I can give 3 inch brush instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "3 inch brush"}}
{"product": "water pump seal kit", "said": "haan ji, mil jayega, delivery nahi karte, aake le jao", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "हाँ बोलिए। नहीं है। घर पर भेज देंगे। दुकान 9 बजे तक खुली है", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "ji boliye, nahi milega bhai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Out of stock. I can give local brake shoe instead. We are open till 10 pm", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local brake shoe"}}
{"product": "50 mm padlock", "said": "woh hum nahi rakhte, dukaan pe aake lena padega", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "thodi der baad call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "It's in stock. Price is 400. We don't deliver", "answer": {"in_stock": true, "price_inr": 400, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "नाही आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी, मिल जाएगा। 60 लगेगा", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "We don't keep that. We can deliver", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "है भाई। 4 पीस पड़े हैं। रेट सौ है। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": 100, "delivery": false, "alternative": null}}
{"product": "bike headlight bulb", "said": "haan bola, nahi aahe, tyachya aivaji LED headlight bulb deu shakto, yeun nyava lagel", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "LED headlight bulb"}}
{"product": "Activa air filter", "said": "malak nahit, nantar phone kara", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "संपलं. 40 mm padlock आहे, चालेल का?. दुकान 10 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm padlock"}}
{"product": "bike headlight bulb", "said": "Yes, we have it. That will be Rs 220. Pickup only", "answer": {"in_stock": true, "price_inr": 220, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji boliye, hai bhai, ₹270, 10 baje ke pehle aana", "answer": {"in_stock": true, "price_inr": 270, "delivery": null, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "ho aahe, 110 la padel", "answer": {"in_stock": true, "price_inr": 110, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "ji boliye, haan ji, mil jayega, 14 piece pade hain, 80 rupaye final", "answer": {"in_stock": true, "price_inr": 80, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "nahi aahe", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "आहे ना. 3 पीस आहेत. घरी पाठवतो", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "Not today, it comes tomorrow. I can give Jupiter clutch plate instead", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Jupiter clutch plate"}}
{"product": "4 inch door hinge", "said": "available hai, 80 lagega, dukaan pe aake lena padega", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "Activa air filter", "said": "ji haan, rakha hai, 3 hi bache hain, 320 lagega", "answer": {"in_stock": true, "price_inr": 320, "delivery": null, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "stock madhe aahe, rate 330 aahe, porga pathavto", "answer": {"in_stock": true, "price_inr": 330, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Out of stock. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही आहे. येऊन न्यावं लागेल", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan bola, haan aahe, 80 la padel, yeun nyava lagel, dukan 8 vajta paryant ughada aahe", "answer": {"in_stock": true, "price_inr": 80, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes, available. We have 18 pieces. It's 780 rupees", "answer": {"in_stock": true, "price_inr": 780, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "Yes sir, we have. 10 rupees", "answer": {"in_stock": true, "price_inr": 10, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "nahi hai, local spark plug hai, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "10W-30 engine oil 1 L", "said": "woh hum nahi rakhte, 10W-30 engine oil 900 ml rakha hai, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "10W-30 engine oil 900 ml"}}
{"product": "4 inch door hinge", "said": "ho, milel, 60 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "हाँ बोलिए। नहीं है। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "wall putty 20 kg", "said": "Yes sir, we have", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "हॅलो, कोण?. संपलं", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Yes, tell me. Yes, available. It's 1450 rupees. I'll send it with our boy", "answer": {"in_stock": true, "price_inr": 1450, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "अभी नहीं है, परसों आएगा", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes sir, we have. Only 19 left. 170 rupees", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "nahi hai, ghar pe bhej denge", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "4 inch door hinge", "said": "hello, kon?, ho, milel, porga pathavto", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "haan bola, ho, milel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "हाँ बोलिए। है भाई। 16 पीस पड़े हैं। डिलीवरी नहीं करते, दुकान पर आकर ले जाइए", "answer": {"in_stock": true, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "2 inch paint brush", "said": "malik abhi nahi hain, baad mein phone karo, dukaan 9 baje tak khuli hai", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "aahe na, 35 rupaye honar", "answer": {"in_stock": true, "price_inr": 35, "delivery": null, "alternative": null}}
{"product": "4 inch door hinge", "said": "Yes, available. Price is 100. Home delivery is available. We are open till 8 pm", "answer": {"in_stock": true, "price_inr": 100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "abhi nahi hai, parso aayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "हां बोला. आम्ही ते ठेवत नाही. 12V 9Ah battery आहे, चालेल का?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "12V 9Ah battery"}}
{"product": "16A MCB", "said": "आहे ना. 180 रुपये. येऊन न्यावं लागेल. दुकान 7 वाजेपर्यंत उघडं आहे", "answer": {"in_stock": true, "price_inr": 180, "delivery": false, "alternative": null}}
{"product": "10 mm nut bolt (pack of 10)", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 socket extension board", "said": "ho, milel, 600 la padel, yeun nyava lagel", "answer": {"in_stock": true, "price_inr": 600, "delivery": false, "alternative": null}}
{"product": "water pump seal kit", "said": "Yes, we have it. 280 rupees. We are open till 7 pm", "answer": {"in_stock": true, "price_inr": 280, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "haan ji, mil jayega, dhai sau ka padega, delivery ho jayegi", "answer": {"in_stock": true, "price_inr": 250, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "Not sure, I'll have to check the godown", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Hold on, let me check", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "हॅलो, कोण?. आहे ना. 5 पीस आहेत. रेट 200 आहे. येऊन न्यावं लागेल", "answer": {"in_stock": true, "price_inr": 200, "delivery": false, "alternative": null}}
{"product": "8 mm drill bit", "said": "woh hum nahi rakhte", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "रुकिए, देखकर बताता हूँ। दुकान 10 बजे तक खुली है", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "Yes, tell me. Yes, available. That will be Rs 170", "answer": {"in_stock": true, "price_inr": 170, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "हैलो, कौन?। हाँ, है। 330 रुपये का है", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hai bhai, 340 ka padega", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "Not today, it comes tomorrow. I can give mechanical seal instead. We don't deliver. Come before 7", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": "mechanical seal"}}
{"product": "1 inch PVC elbow", "said": "woh hum nahi rakhte, delivery nahi karte, aake le jao", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "wall putty 20 kg", "said": "available hai", "answer": {"in_stock": true, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "12V 7Ah battery", "said": "haan ji, mil jayega, 13 piece pade hain, ₹1100, ghar pe bhej denge", "answer": {"in_stock": true, "price_inr": 1100, "delivery": true, "alternative": null}}
{"product": "silicone sealant", "said": "Hello?. Yes, available. We can deliver", "answer": {"in_stock": true, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "hello, kaun?, hai bhai, rate 340 hai", "answer": {"in_stock": true, "price_inr": 340, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "sampla, 60 mm padlock aahe, chalel ka?, ghari pathavto", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "60 mm padlock"}}
{"product": "9W LED bulb", "said": "नहीं है। उसकी जगह 7W LED bulb दे सकते हैं। होम डिलीवरी हो जाएगी", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "7W LED bulb"}}
{"product": "2 inch paint brush", "said": "haan bolo, nahi milega bhai, home delivery hai, dukaan 7 baje tak khuli hai", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "tank float valve", "said": "haan, stock mein hai, 17 piece pade hain, ₹330", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "1 inch PVC elbow", "said": "हाँ, है। चालीस लगेगा। होम डिलीवरी हो जाएगी", "answer": {"in_stock": true, "price_inr": 40, "delivery": true, "alternative": null}}
'''.strip().splitlines()]
TRAIN_SET = globals().get("TRAIN_SET", "v2")
TRAIN_DATA = TRAIN_V2 + REAL_XPQA if TRAIN_SET == "v2" else TRAIN_V1
TEST = [json.loads(line) for line in r'''
{"product": "10 mm nut bolt (pack of 10)", "said": "haan bhai mil jayega, packet ka 60 lagega. aap aake le jao, bhejne wala koi nahi hai", "answer": {"in_stock": true, "price_inr": 60, "delivery": false, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "nahi hai abhi, 500 gram wala hai chalega toh", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "500 gram"}}
{"product": "16A MCB", "said": "Yes we have, 220 rupees. Delivery also possible in the evening.", "answer": {"in_stock": true, "price_inr": 220, "delivery": true, "alternative": null}}
{"product": "Activa clutch plate", "said": "ho aahe, 850 la padel. ghari pathavto", "answer": {"in_stock": true, "price_inr": 850, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "संपली आहे, सोमवारी येईल", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "9W LED bulb", "said": "हाँ जी है, 90 रुपये का है", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "teflon tape", "said": "kaun? achha tape... ruko beta ko puchta hun, baad mein call karo", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Pulsar 150 spark plug", "said": "original khatam, local spark plug hai chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "local spark plug"}}
{"product": "wall putty 20 kg", "said": "aahe na, 20 kilo cha bag 780. delivery nahi, tempo karun nyava lagel", "answer": {"in_stock": true, "price_inr": 780, "delivery": false, "alternative": null}}
{"product": "50 mm padlock", "said": "Sorry sir, that size is finished. 40 mm is there.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "40 mm"}}
{"product": "2 inch paint brush", "said": "है, 45 का एक। दुकान 9 बजे तक खुली है", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "fan capacitor 2.5 mfd", "said": "haan hai, 70. home delivery nahi karte", "answer": {"in_stock": true, "price_inr": 70, "delivery": false, "alternative": null}}
{"product": "1.5 sq mm copper wire, 90 m roll", "said": "Available hai sir, full roll 1850 ka. Ghar pe bhijwa denge", "answer": {"in_stock": true, "price_inr": 1850, "delivery": true, "alternative": null}}
{"product": "Activa air filter", "said": "थांबा हं, मालक जेवायला गेलेत. अर्ध्या तासाने फोन करा", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "10W-30 engine oil 1 L", "said": "10W-30 nahi rakhte, 20W-40 milega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20W-40"}}
{"product": "bike headlight bulb", "said": "yes available, ₹150, but pickup only", "answer": {"in_stock": true, "price_inr": 150, "delivery": false, "alternative": null}}
{"product": "4 socket extension board", "said": "हो आहे, ४५० रुपये", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "tank float valve", "said": "nahi hai bhai, kal shaam tak aa jayega", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "8 mm drill bit", "said": "hai, 3 piece bache hain, 60 ka ek. ladka chhod dega ghar pe", "answer": {"in_stock": true, "price_inr": 60, "delivery": true, "alternative": null}}
{"product": "Splendor brake shoe", "said": "Splendor ka hai na, 260. dukanat yeun ghya", "answer": {"in_stock": true, "price_inr": 260, "delivery": false, "alternative": null}}
{"product": "silicone sealant", "said": "We don't keep sealant. Try M-seal, we have that.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "M-seal"}}
{"product": "1 inch PVC elbow", "said": "हाँ है, बीस रुपये का एक", "answer": {"in_stock": true, "price_inr": 20, "delivery": null, "alternative": null}}
{"product": "water pump seal kit", "said": "hello? haan bolo... seal kit? hai, 300. delivery hoti hai", "answer": {"in_stock": true, "price_inr": 300, "delivery": true, "alternative": null}}
{"product": "Fevicol SH 1 kg", "said": "aahe, 390. aamhi delivery deto pan 50 rupaye extra", "answer": {"in_stock": true, "price_inr": 390, "delivery": true, "alternative": null}}
{"product": "12V 7Ah battery", "said": "Let me check... yes one piece is left. 1350 rupees.", "answer": {"in_stock": true, "price_inr": 1350, "delivery": null, "alternative": null}}
{"product": "16A MCB", "said": "MCB 16 amp? nahi hai, 20 amp wala hai", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "20 amp"}}
{"product": "9W LED bulb", "said": "पता नहीं, गोदाम में देखना पड़ेगा। शाम को फ़ोन करना", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Activa clutch plate", "said": "नाही, आम्ही ॲक्टिव्हाचे पार्ट ठेवत नाही", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "50 mm padlock", "said": "haan hai, 350 ka. 7 baje tak aa jao, delivery nahi hai", "answer": {"in_stock": true, "price_inr": 350, "delivery": false, "alternative": null}}
{"product": "teflon tape", "said": "hai, 15 rupaye. ek kaam karo, 10 roll le lo toh 12 ka laga dunga", "answer": {"in_stock": true, "price_inr": 15, "delivery": null, "alternative": null}}
'''.strip().splitlines()]
HARD = [json.loads(line) for line in r'''
{"product": "TVS Jupiter tubeless tyre 90/90-12", "said": "Jupiter ka tyre ek hi tha, woh subah bik gaya, ab dukaan mein ek bhi nahi bacha. Parso tak naya maal aayega. Activa wala tyre same size ka pada hai, MRF ka, woh chalega kya?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Activa wala tyre"}}
{"product": "Godrej mortise lock 6 lever", "said": "Pehle ka rate sawa gyarah sau tha, ab company ne badha ke barah sau kar diya hai. Ek hi piece pada hai, chahiye toh abhi rakhwa lo.", "answer": {"in_stock": true, "price_inr": 1200, "delivery": null, "alternative": null}}
{"product": "Hero Splendor chain kit", "said": "Sir, I joined only last week, so the owner will tell you the price and all. He is at the bank now and will be back after 4. Please call then or tomorrow.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia screwdriver set (6 pcs)", "said": "The set is there... sorry, that is the 4 piece set. 6 piece we don't have. 4 piece is there at 180, will you take that?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "4 piece set"}}
{"product": "Maruti Alto oil filter", "said": "Alto ka oil filter ek hi piece pada hai, 180 ka. Do chahiye the toh kal tak mangwa dunga. Ladka ghar tak chhod dega, uska 40 rupaye alag se lagega.", "answer": {"in_stock": true, "price_inr": 180, "delivery": true, "alternative": null}}
{"product": "Stanley 300 g claw hammer", "said": "Claw hammer 300 gram is there. Rs 240 only, handle is wooden. Sir, you are breaking... yes 240. And you have to collect from the shop, we do not send anything home.", "answer": {"in_stock": true, "price_inr": 240, "delivery": false, "alternative": null}}
{"product": "Bajaj mixer grinder carbon brush", "said": "ब्रश तो हैं, लेकिन बजाज मिक्सर वाला नहीं, फिलिप्स वाला पड़ा है, वो दे दूँ? बजाज वाला शनिवार तक आएगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "फिलिप्स वाला"}}
{"product": "Havells Crabtree 16A 3-pin socket", "said": "Last month it was 85, now the new rate is 92 after GST change. Stock is there, five-six pieces.", "answer": {"in_stock": true, "price_inr": 92, "delivery": null, "alternative": null}}
{"product": "Anchor Penta 6A two-way switch", "said": "Yes sir, available only. One piece is Rs. 38/- only, but if you take the full box of 10 then 320, so 32 each.", "answer": {"in_stock": true, "price_inr": 38, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 chain kit", "said": "चेन किट है... अरे नहीं, वो ऍवेंजर वाला है, आपके काम का नहीं। पल्सर १५० का किट तो पिछले महीने से आया ही नहीं।", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "4 inch cutting wheel (pack of 5)", "said": "Cutting wheel cha pack aahe, 5 cha pack 90 la. Pan mala sang, tumhi site var kiti vaparta? Roj vaparat asal tar 25 cha box ghya, to 380 la hoto, jasta parvadto.", "answer": {"in_stock": true, "price_inr": 90, "delivery": null, "alternative": null}}
{"product": "Syska 20W LED batten", "said": "Syska ka batten hai, 260 mein. Wipro ka 310 mein milta hai, woh chahiye ho toh bata dena, par aapne Syska kaha tha na.", "answer": {"in_stock": true, "price_inr": 260, "delivery": null, "alternative": null}}
{"product": "Royal Enfield Bullet 350 clutch plate set", "said": "Malak aata jevayla gele aahet, mala rate pan mahit nahi ani stock pan baghayla lagel. Tumhi dupari don chya sumaras call kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Nylon cable tie 8 inch (pack of 100)", "said": "पाकीट मिळेल, १२० ला... नाही नाही, आता १४० झालं, काल रेट वाढला.", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Distilled water 5 L can", "said": "5 litre cha can nahiye ho, 1 litre chya bottle aahet, 25 rupaye chi ek. Chalel ka tumhala?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "1 litre chya bottle"}}
{"product": "Crompton 1200 mm ceiling fan", "said": "Hoy, 1200 mm cha pankha ready aahe, white colour madhe. Diwali offer chalu aahe mhanun pahile 2400 la hota, aata 2150 la deto, ani ekach piece urlay.", "answer": {"in_stock": true, "price_inr": 2150, "delivery": null, "alternative": null}}
{"product": "Fischer 8 mm plastic wall plug (pack of 100)", "said": "होय, १०० चं पाकीट आहे, ६० ला मिळतं. सुटे हवे असतील तर १ रुपयाला एक देतो.", "answer": {"in_stock": true, "price_inr": 60, "delivery": null, "alternative": null}}
{"product": "Crompton 9 inch exhaust fan", "said": "9 inch wala toh is baar aaya hi nahi, 12 inch wala pada hai, woh bhej dun? Delivery toh hum karte hi hain, kal shaam tak ho jayegi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "12 inch wala"}}
{"product": "Eveready 3 cell torch", "said": "टॉर्च आहे, पण आजच किंमत वाढली. आधी १८० ला होता, आता दोनशे दहा झाला. एक पीस पडलाय फक्त.", "answer": {"in_stock": true, "price_inr": 210, "delivery": null, "alternative": null}}
{"product": "Luminous 150Ah inverter battery", "said": "Malik se baat karo bhai, rate wahi batate hain. Main toh bas counter pe baitha hun, mujhe stock bhi nahi pata. Kal subah 11 ke baad phone karna.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Amaron 35Ah car battery", "said": "Haan 35 wali hai... matlab thi, kal tak. Kal hi do piece nikal gaye, ab ek bhi nahi bacha.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "SDS plus 12 mm drill bit", "said": "12 mm cha SDS bit nahiye ajun, order dila aahe. Aala ki aamhi tumchya site var gadi pathavu, delivery cha charge nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Honda Shine chain sprocket kit", "said": "Shine cha kit aala hota, pan kalach don piece gele. Aata ek pan urla nahi. Udya sakali gaadi yeil. Pathvaychi soy nahiye, tumhi yeun neta ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Cera 15 mm CP angle cock", "said": "Cera चा नाहीये, जॅग्वारचा आहे, चालेल ना? त्याची किंमत ३५० आहे.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "जॅग्वारचा"}}
{"product": "Maruti Swift Dzire wiper blade (pair)", "said": "Dzire cha wiper aahe, pair madhe 380 la. Pan ekach pair urlay mhanun lavkar ghya, nahitar udya sampel.", "answer": {"in_stock": true, "price_inr": 380, "delivery": null, "alternative": null}}
{"product": "Anchor Roma 6A switch", "said": "Roma ka 6 amp wala switch? Woh toh paintalis ka aata hai, kal hi naya maal utra hai, abhi 2 dabbe pade hain counter ke neeche.", "answer": {"in_stock": true, "price_inr": 45, "delivery": null, "alternative": null}}
{"product": "Bajaj Pulsar 150 front brake pad", "said": "Pulsar cha pad aahe, pan to 220 cha. 150 cha nahiye amchyakade, ani 220 cha tumchya gadi la basnar nahi.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Ceiling fan regulator (step type)", "said": "Step wala regulator hai... ruko, arre woh toh electronic wala hai. Step wala toh pichhle mahine hi khatam hua, electronic wala 180 mein deta hun, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "electronic wala"}}
{"product": "UltraTech cement 50 kg bag", "said": "Cement aaj hi gaadi se utra hai, 50 kilo ka bag 395 ka. 20 bag se upar lo toh 385 laga dunga. Ghar tak nahi bhejte hum, khud tempo lekar aana padega.", "answer": {"in_stock": true, "price_inr": 395, "delivery": false, "alternative": null}}
{"product": "Bajaj Platina clutch cable (original)", "said": "Platina ka clutch cable rakha hai. Local wala nabbe ka milta hai, original Bajaj ka sawa do sau mein. Aapko original chahiye na, toh woh sawa do sau.", "answer": {"in_stock": true, "price_inr": 225, "delivery": null, "alternative": null}}
{"product": "Wipro 20W LED tube light", "said": "Tube aahe, paune donshe la ek. Choke nako lagat, direct lavta yeto.", "answer": {"in_stock": true, "price_inr": 175, "delivery": null, "alternative": null}}
{"product": "Orient 400 mm pedestal fan", "said": "आज रेट नहीं बता सकता, कल बोलना। सेठ जी बैंक गए हैं और स्टॉक की लिस्ट उन्हीं के पास है।", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Bosch 10 mm hammer drill machine", "said": "Bosch ka toh stock mein hai hi nahi, Stanley ka 10 mm wala pada hai, woh 2200 mein de dunga, chalega?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Stanley ka 10 mm wala"}}
{"product": "Bosch GWS 600 angle grinder", "said": "It is not that we don't keep Bosch, grinders are always there. Just the 600 watt one got sold today, the 850 watt is there at 4200. Shall I keep that for you?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "850 watt"}}
{"product": "Cera half inch CP bib cock", "said": "Bib cock aahe, Cera cha ek doshe pannas la. Local cha pan aahe, to ekshe vees la.", "answer": {"in_stock": true, "price_inr": 250, "delivery": null, "alternative": null}}
{"product": "Havells 4-way MCB box", "said": "४ वे चा बॉक्स संपलाय, पण आम्ही घरपोच करतो, फक्त माल आला की.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Orient 16 inch table fan blade", "said": "Blade aaj nahi milega, jo the woh sab Sunday ko nikal gaye. Naya maal Tuesday ko aayega, 3 baje tak phone kar lena.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Taparia 8-piece spanner set", "said": "No sir, the last Taparia set went yesterday evening. New stock will come on Thursday after 11. Whenever it comes you have to collect, we do not send things home. Other brand also we don't keep.", "answer": {"in_stock": false, "price_inr": null, "delivery": false, "alternative": null}}
{"product": "Havells musical door bell", "said": "Hoy aahe... ek minute, mi chukicha bolto, to musical nahi, buzzer aahe. Musical bell nahiye, buzzer deu ka 150 la?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "buzzer"}}
{"product": "Araldite 15 g tube", "said": "अरल्डाइट? आज सुबह आख़िरी ट्यूब चली गई। कल दोपहर २ बजे के बाद नया बॉक्स आएगा, घर तक भिजवा दूँगा।", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": null}}
{"product": "Stanley 5 m measuring tape", "said": "Aahe... nahi thamba, to Freemans cha aahe, Stanley cha nahiye. Freemans cha 5 meter cha tape 120 la deu ka?", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "Freemans"}}
{"product": "Tata Ace coolant 1 L", "said": "Mi Nilesh bolto, malak kalach gavi gele. Rate te yeun sangtil. Stock baddal mala kahi mahit nahi, ya aathvadyat parat phone kara.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Dr. Fixit LW+ 1 litre", "said": "LW+ one litre bottle, yes. MRP is three twenty but I will give for two ninety, final. Hundred rupees extra if you want it sent to your site.", "answer": {"in_stock": true, "price_inr": 290, "delivery": true, "alternative": null}}
{"product": "Havells 25A DP MCB", "said": "DP 25 amp ka hai sir, 540 mein. Mera number likh lo 9822334455, kuch aur chahiye toh WhatsApp kar dena.", "answer": {"in_stock": true, "price_inr": 540, "delivery": null, "alternative": null}}
{"product": "Hero Splendor Plus side mirror (left)", "said": "Left cha mirror aahe, right cha nahiye. Tumhala left cha pahije na? 130 la ek.", "answer": {"in_stock": true, "price_inr": 130, "delivery": null, "alternative": null}}
{"product": "Asian Paints Tractor Emulsion 4 L (white)", "said": "White 4 litre abhi bacha nahi, cream 4 litre pada hai, usse bhi kaam chal jata hai. Uska 780 padta hai.", "answer": {"in_stock": false, "price_inr": null, "delivery": null, "alternative": "cream 4 litre"}}
{"product": "Hero Passion Pro tail lamp", "said": "टेल लैंप पड़ा है, सवा दो सौ में दे दूँगा, बल्ब भी लगा हुआ आता है। घर तक भेजने का इंतज़ाम नहीं रखते, खुद उठाना पड़ेगा।", "answer": {"in_stock": true, "price_inr": 225, "delivery": false, "alternative": null}}
{"product": "PVC conduit pipe 20 mm (3 m piece)", "said": "ISI conduit pipe, yes. Per piece 25 rupees, if you take full bundle of 10 then 230. Delivery I will check with the driver and tell you, not sure.", "answer": {"in_stock": true, "price_inr": 25, "delivery": null, "alternative": null}}
{"product": "Hacksaw blade 12 inch (pack of 10)", "said": "हाँ मिल जाएगा, १० वाले पैकेट के १४० रुपये ले लेता हूँ। पर अगर आप एक-दो ब्लेड लोगे तो १८ का एक पड़ेगा।", "answer": {"in_stock": true, "price_inr": 140, "delivery": null, "alternative": null}}
{"product": "Philips 18W LED panel light (round)", "said": "Round 18 watt cha panel aahe, 480 la. GST bill pahije asel tar 18 tax vegla lagto.", "answer": {"in_stock": true, "price_inr": 480, "delivery": null, "alternative": null}}
{"product": "Activa 6G CVT belt", "said": "रेट आज नाही सांगता येणार, मालक बाहेर गेलेत. उद्या फोन करा, मग स्टॉक पण बघून सांगतो.", "answer": {"in_stock": null, "price_inr": null, "delivery": null, "alternative": null}}
{"product": "Havells 63A 2-pole ELCB", "said": "2 pole cha ELCB aahe ho, 1450 la dein. 4 pole cha pan aahe, to 3100 la, pan to tumhala nako asel na?", "answer": {"in_stock": true, "price_inr": 1450, "delivery": null, "alternative": null}}
{"product": "1500W water heater element", "said": "1500 watt is finished since morning, sir. 2000 watt is there, can you use that? Our boy can drop it at your place, no extra charge.", "answer": {"in_stock": false, "price_inr": null, "delivery": true, "alternative": "2000 watt"}}
{"product": "Royal Enfield Classic 350 air filter (original)", "said": "Original or local? We keep both. Original RE one is 450, local is 160. Original is in stock, only 2 pieces, local is plenty.", "answer": {"in_stock": true, "price_inr": 450, "delivery": null, "alternative": null}}
{"product": "Servo 80W-90 gear oil 1 L", "said": "Gear oil? Awaaz kat rahi hai... haan ab sunai diya. Servo ka 80W-90 hai, 3 litre ka can 900 ka aata hai aur 1 litre ka 330 ka.", "answer": {"in_stock": true, "price_inr": 330, "delivery": null, "alternative": null}}
{"product": "ESAB 3.15 mm welding rod (1 kg packet)", "said": "3.15 cha rod aahe ho, 1 kg cha packet 190 la. Loose pan deto, 20 rod cha 70 la. Pathvaychi soy nahiye, tumchya manus la pathva.", "answer": {"in_stock": true, "price_inr": 190, "delivery": false, "alternative": null}}
'''.strip().splitlines()]
FRESH = []
SETS = {'test': TEST, 'hard': HARD, **({'fresh': FRESH} if FRESH else {})}
print(TRAIN_SET, len(TRAIN_DATA), "training rows;", {k: len(v) for k, v in SETS.items()}, "test calls")

In [ ]:
#@title 5. Load the base model (about 2–3 minutes)
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import importlib.metadata as md
import torch

print({p: md.version(p) for p in ("unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft")})

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=1024, load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
# LoRA: train a small add-on (about 1% of the weights) instead of the whole model.
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407,
)

In [ ]:
#@title 6. Test the model BEFORE training
def ask(product, said):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(product, said)}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_tests(label):
    FastLanguageModel.for_inference(model)
    try:
        results = evaluate(ask, SETS)
    finally:
        FastLanguageModel.for_training(model)
    show(label, results)
    return results


before = run_tests("BEFORE training")

In [ ]:
#@title 7. Train (about 10–15 minutes)
from datasets import Dataset
from trl import SFTConfig, SFTTrainer


def to_text(ex):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_message(ex["product"], ex["said"])},
                {"role": "assistant", "content": answer_text(ex["answer"])}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


dataset = Dataset.from_list([to_text(ex) for ex in TRAIN_DATA])
print(dataset[0]["text"])

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", max_length=1024,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, lr_scheduler_type="linear",
        optim="adamw_8bit", weight_decay=0.01, logging_steps=10, seed=3407,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        output_dir="outputs", report_to="none",
    ),
)
# Learn only the answers, not the questions.
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime'] / 60:.1f} min, loss {stats.metrics['train_loss']:.3f}")

In [ ]:
#@title 8. Test the model AFTER training
after = run_tests("AFTER training")
payload = results_payload(MODEL_NAME, EPOCHS, before, after)
payload.update(train_set=TRAIN_SET, train_rows=len(TRAIN_DATA))
print("\n== Before vs after (all 4 fields right)")
for name, res in payload["sets"].items():
    print(f"   {name:5s} {res['before']['all_correct']:4.0%} -> {res['after']['all_correct']:4.0%}  ({res['examples']} calls)")
print("\nRESULTS_JSON", json.dumps({k: {n: r[k] for n, r in payload["sets"].items()} for k in ("before", "after")}))
if ON_KAGGLE:  # run_on_kaggle.py downloads this
    with open("/kaggle/working/results.json", "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=1)

In [ ]:
#@title 9. Try your own sentence
product = "Activa clutch plate"  #@param {type:"string"}
said = "haan hai, 650 ka padega, ghar pe bhej denge"  #@param {type:"string"}
FastLanguageModel.for_inference(model)
print(ask(product, said))

In [ ]:
#@title 10. Save your model: Google Drive on Colab, the Output tab on Kaggle (the Ollama file takes 10–15 minutes)
import glob, re

lora_dir = f"{WORK}/{OUTPUT_NAME}-lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
if SAVE_DIR:
    os.makedirs(SAVE_DIR, exist_ok=True)
    shutil.copytree(lora_dir, f"{SAVE_DIR}/lora", dirs_exist_ok=True)
    print("Saved the LoRA add-on to", f"{SAVE_DIR}/lora")

if MAKE_OLLAMA_FILE:
    gguf_dir = f"{WORK}/{OUTPUT_NAME}"
    result = model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
    files = (result or {}).get("gguf_files") if isinstance(result, dict) else None
    files = files or glob.glob(f"{gguf_dir}*/**/*.gguf", recursive=True)
    if not files:
        raise RuntimeError("No .gguf file was made. The LoRA add-on is still saved; send Claude the error above.")
    q4 = [f for f in files if "q4_k_m" in os.path.basename(f).lower()] or sorted(files, key=os.path.getsize)
    gguf = q4[0]
    name = f"{OUTPUT_NAME}.Q4_K_M.gguf"

    modelfile = make_modelfile(name)
    modelfile_path = os.path.join(os.path.dirname(gguf), "Modelfile.wimb")  # gguf_dir itself may be cleaned up
    with open(modelfile_path, "w", encoding="utf-8") as fh:
        fh.write(modelfile)
    print(modelfile)

    if SAVE_DIR:
        shutil.copy(gguf, f"{SAVE_DIR}/{name}")
        shutil.copy(modelfile_path, f"{SAVE_DIR}/Modelfile")

if SAVE_DIR:
    for path in sorted(glob.glob(f"{SAVE_DIR}/*")):
        size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else 0
        print(f"   {path}  {size:,.0f} MB" if size else f"   {path}/")

In [ ]:
#@title 11. Test the exported GGUF file itself (CPU, up to 25 minutes; the answers Ollama would give)
import socket, subprocess, time, urllib.request

gguf_report = {"error": "not run"}
server = None
try:
    if not MAKE_OLLAMA_FILE:
        raise RuntimeError("MAKE_OLLAMA_FILE is off, so there is no GGUF file to test")
    binary = subprocess.run(
        ["bash", "-c", "find / -xdev -name llama-server -type f -perm -u+x 2>/dev/null | head -1"],
        capture_output=True, text=True).stdout.strip()
    if not binary:
        raise RuntimeError("llama-server was not found (Unsloth builds it while making the GGUF)")
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]
    server = subprocess.Popen([binary, "-m", gguf, "--port", str(port), "--host", "127.0.0.1", "-c", "2048",
                               "-t", str(os.cpu_count() or 2), "--jinja"],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    base = f"http://127.0.0.1:{port}"
    for _ in range(120):  # up to 6 minutes to load the model
        if server.poll() is not None:
            raise RuntimeError(f"llama-server stopped at start-up (exit code {server.returncode})")
        try:
            urllib.request.urlopen(base + "/health", timeout=5).read()
            break
        except Exception:
            time.sleep(3)
    else:
        raise RuntimeError("llama-server did not become ready")

    def ask_gguf(product, said):
        body = json.dumps({"messages": [{"role": "system", "content": SYSTEM},
                                        {"role": "user", "content": user_message(product, said)}],
                           "temperature": 0, "max_tokens": 96}).encode()
        req = urllib.request.Request(base + "/v1/chat/completions", body, {"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=300) as resp:
            return json.load(resp)["choices"][0]["message"]["content"].strip()

    started = time.time()
    gguf_results = evaluate(ask_gguf, SETS, deadline=started + 25 * 60)
    show("GGUF Q4_K_M", gguf_results)
    gguf_report = {"gguf": os.path.basename(gguf), "seconds": round(time.time() - started),
                   "sets": {n: {"examples": r["examples"], "of": r["of"],
                                **({k: round(r[k], 3) for k in KEYS} if r["examples"] else {}),
                                "mistakes": r["mistakes"], "replies": r["replies"]}
                            for n, r in gguf_results.items()}}
except Exception as e:  # the main results are already saved; this test must never fail the run
    print("GGUF test failed:", repr(e))
    gguf_report = {"error": repr(e)}
finally:
    if server is not None:
        server.terminate()
if ON_KAGGLE:
    with open("/kaggle/working/results_gguf.json", "w", encoding="utf-8") as fh:
        json.dump(gguf_report, fh, ensure_ascii=False, indent=1)

## Done. Next steps

**1. Save this notebook to GitHub** so Claude can read the results:
**File → Save a copy in GitHub** → `pratik9yadav/yok`, branch `main`,
path `train_colab.ipynb`.

**2. Run your model on your PC**
1. Install Ollama from ollama.com.
2. From Google Drive, download the `wimb-shop-model` folder (the `.gguf` file and `Modelfile`).
3. Open a terminal in that folder and run:
```
ollama create wimb-shop -f Modelfile
ollama run wimb-shop "Product: 16A MCB
Shopkeeper: haan hai, 220 ka, delivery nahi"
```